# 04 Scenario Simulation & Strategic Impact

**Context:**
Notebook 03 statistically validated findings, proving that margin destruction is systemic and driven by specific structural failures (zombie SKUs, extreme profit concentration, chronic operational drag).

**Purpose:**
This notebook translates those validated audit findings into quantified financial projections under three operational stress scenarios. I will establish the current baseline, simulate the elimination of "Cut Candidates," stress-test the core profit drivers, and project the cost of compounded operational failure.

## 1. Environment Setup
Load the classified catalog data and the ground truth operational metrics.

In [1]:
import pandas as pd
import numpy as np

# Load data
classifications_path = '../data/processed/quadrant_classifications.csv'
ground_truth_path = '../data/processed/ground_truth_master.csv'

# Read CSVs
df_class = pd.read_csv(classifications_path)
df_gt = pd.read_csv(ground_truth_path)

# Merge the quadrant and risk score data into the ground truth dataframe
# This gives all revenue/profit data + the operational segments in one place
df = pd.merge(
    df_gt,
    df_class[['product_card_id', 'quadrant', 'composite_risk_score']],
    on='product_card_id',
    how='left'
)

print(f"Simulation dataset loaded: {df.shape[0]} products ready.")

Simulation dataset loaded: 118 products ready.


## 2. Baseline Establishment
Before simulating any operational changes, I must lock the current state as reference point. This baseline quantifies exactly what the business is currently generating, and critically, how much of that profit is reliant on the 8 core SKUs identified in Phase 3.

In [2]:
# Calculating Macro Baselines
baseline_revenue = df['total_revenue'].sum()
baseline_profit = df['total_profit'].sum()
baseline_margin = baseline_profit / baseline_revenue
total_skus = len(df)

# Isolate the Core 8 SKUs (Identified via upper IQR bound in Notebook 03)
core_8_df = df.sort_values(by='total_profit', ascending=False).head(8)
core_8_profit = core_8_df['total_profit'].sum()
core_8_revenue = core_8_df['total_revenue'].sum()
core_8_profit_pct = core_8_profit / baseline_profit

# Print Clean Summary Table
print("CURRENT STATE BASELINE")
print(f"Total Active SKUs:          {total_skus}")
print(f"Total Catalog Revenue:      ${baseline_revenue:,.2f}")
print(f"Total Catalog Profit:       ${baseline_profit:,.2f}")
print(f"Average Net Margin:         {baseline_margin:.2%}\n")
print("PROFIT CONCENTRATION")
print(f"Core 8 SKU Profit:          ${core_8_profit:,.2f}")
print(f"Core 8 SKU Revenue:         ${core_8_revenue:,.2f}")
print(f"Core Reliance Metric:       {core_8_profit_pct:.1%} of Total Profit")

CURRENT STATE BASELINE
Total Active SKUs:          118
Total Catalog Revenue:      $35,214,428.98
Total Catalog Profit:       $3,806,420.63
Average Net Margin:         10.81%

PROFIT CONCENTRATION
Core 8 SKU Profit:          $3,216,447.46
Core 8 SKU Revenue:         $29,834,820.73
Core Reliance Metric:       84.5% of Total Profit


## 3. Scenario Definitions and Assumptions
To translate findings into actionable financial impact, I will stress-test the catalog under three distinct operational scenarios:

* **Scenario A: Zombie SKU Elimination:** Immediate removal of all 28 "Cut Candidates" from the catalog. Assumption: 100% of associated revenue and profit is lost, but 100% of associated operational bandwidth (inventory, fulfillment of late orders) is freed.
* **Scenario B: Core SKU Disruption:** A 20% margin compression applied exclusively to the 8 core profit drivers. Assumption: Represents a severe external shock (supplier price hike, increased fulfillment costs) to most vulnerable concentration point.
* **Scenario C: Compounded Operational Stress:** A 10% universal increase in discount dependency combined with a 15% increase in late delivery risk. Assumption: Represents the systemic decay of the supply chain. Impact is projected directionally using the validated Composite Risk Score correlation.

## 4. Scenario A: Zombie SKU Elimination
Simulate the financial impact of amputating the 28 products identified in the "Cut Candidates" quadrant. These products suffer from weak unit economics and severe operational delivery failure.

In [3]:
# Isolate the catalog WITHOUT the Cut Candidates
scenario_a_df = df[df['quadrant'] != 'Cut Candidates'].copy()

# Calculate Scenario A metrics
scen_a_revenue = scenario_a_df['total_revenue'].sum()
scen_a_profit = scenario_a_df['total_profit'].sum()
scen_a_margin = scen_a_profit / scen_a_revenue
skus_removed = total_skus - len(scenario_a_df)

# Calculate Deltas
rev_retained_pct = scen_a_revenue / baseline_revenue
profit_retained_pct = scen_a_profit / baseline_profit
margin_change = scen_a_margin - baseline_margin

print("SCENARIO A: ZOMBIE SKU ELIMINATION")
print(f"Removed {skus_removed} SKUs (Cut Candidates quadrant)")
print(f"Projected Revenue:      ${scen_a_revenue:,.2f} ({rev_retained_pct:.2%} retained)")
print(f"Projected Profit:       ${scen_a_profit:,.2f} ({profit_retained_pct:.2%} retained)")
print(f"New Catalog Margin:     {scen_a_margin:.2%} (Improvement of {(margin_change * 100):.2f} pts)\n")

print("Strategic Takeaway:")
print(f"By cutting {skus_removed} failing products, the business retains {profit_retained_pct:.2%} of its profit while freeing up nearly 25% of its supply chain bandwidth.")

SCENARIO A: ZOMBIE SKU ELIMINATION
Removed 28 SKUs (Cut Candidates quadrant)
Projected Revenue:      $34,450,986.89 (97.83% retained)
Projected Profit:       $3,739,117.79 (98.23% retained)
New Catalog Margin:     10.85% (Improvement of 0.04 pts)

Strategic Takeaway:
By cutting 28 failing products, the business retains 98.23% of its profit while freeing up nearly 25% of its supply chain bandwidth.


**Business Interpretation:**
* **Addition by Subtraction:** Scenario A proves that the "Cut Candidates" are operational ghosts. Amputating nearly a quarter of the entire physical catalog (28 SKUs) costs the business less than 2% of its total profit and revenue. 
* **The Bandwidth Dividend:** The true value of this scenario isn't the 0.04 point margin improvement; it is the 25% of supply chain bandwidth that is instantly freed. The capital previously tied up in inventory, the warehouse space, and the customer service hours spent apologizing for the >58% late delivery rates of these specific products can now be reallocated to the core catalog.

## 5. Scenario B: Core SKU Disruption
Simulate a 20% margin compression (profit loss) applied exclusively to the 8 core profit-driving SKUs. This models the financial vulnerability of extreme profit concentration (Finding F-013). Revenue remains constant, but the cost to deliver those top items increases.

In [5]:
# The core 8 profit was calculated in Section 2
compression_rate = 0.20

# Calculate the exact dollar loss from compressing the core 8
core_8_profit_loss = core_8_profit * compression_rate

# Apply loss to the baseline
scen_b_revenue = baseline_revenue  # Revenue stays the same, costs increased
scen_b_profit = baseline_profit - core_8_profit_loss
scen_b_margin = scen_b_profit / scen_b_revenue

# Calculate impact percentages
profit_destroyed_pct = core_8_profit_loss / baseline_profit

print("SCENARIO B: CORE SKU DISRUPTION")
print(f"20% profit compression on the 8 core SKUs.")
print(f"Projected Revenue:      ${scen_b_revenue:,.2f} (No change)")
print(f"Projected Profit:       ${scen_b_profit:,.2f} ({1 - profit_destroyed_pct:.2%} retained)")
print(f"New Catalog Margin:     {scen_b_margin:.2%} (Drop of {((baseline_margin - scen_b_margin) * 100):.2f} pts)\n")

print("Strategic Takeaway:")
print(f"A 20% disruption to just 8 products destroys ${core_8_profit_loss:,.2f} in net profit, wiping out {profit_destroyed_pct:.2%} of the company's entire bottom line.")

SCENARIO B: CORE SKU DISRUPTION
20% profit compression on the 8 core SKUs.
Projected Revenue:      $35,214,428.98 (No change)
Projected Profit:       $3,163,131.14 (83.10% retained)
New Catalog Margin:     8.98% (Drop of 1.83 pts)

Strategic Takeaway:
A 20% disruption to just 8 products destroys $643,289.49 in net profit, wiping out 16.90% of the company's entire bottom line.


**Business Interpretation:**
* **The Fragility of Concentration:** Scenario B proves that the business's extreme profit concentration is a massive structural liability. Because 80%+ of the profit is tied to just 8 SKUs, a localized 20% margin compression on those specific items instantly vaporizes 17% of the entire company's bottom line (over $643,000). 
* **The Illusion of Catalog Safety:** The founder might assume that having a 118-product catalog protects the business from isolated supply chain shocks. This scenario proves the opposite. If the supplier for the "Field & Stream Sportsman Safe" or the "Perfect Fitness Rip Deck" raises prices, the other 110 products cannot generate enough aggregate margin to cover the loss. The catalog is wide, but the safety net is dangerously narrow.

## 6. Scenario C: Compounded Operational Stress
Apply a 10% increase in discount dependency combined with a 15% increase in late delivery risk across the full catalog. 

*Methodology Note (DL-013):* I project this impact directionally using the validated Spearman correlation (0.57) between the Composite Risk Score and Net Margin, proving the cost of allowing chronic operational drag to compound.

In [6]:
# Directional Impact Model parameters
discount_stress = 0.10
delivery_stress = 0.15
spearman_rho = 0.57 # Validated in Notebook 03

# Calculate the average risk vector increase
# Delivery and Discount each make up 1/3 of the Composite Risk Score weighting
composite_risk_increase = (discount_stress * 0.333) + (delivery_stress * 0.333)

# Directional margin decay = Risk Increase * Correlation Strength
margin_decay_rate = composite_risk_increase * spearman_rho

# Calculate Scenario C projections
scen_c_margin = baseline_margin * (1 - margin_decay_rate)
scen_c_revenue = baseline_revenue # Top-line remains constant, bottom-line erodes
scen_c_profit = scen_c_revenue * scen_c_margin
scen_c_profit_loss = baseline_profit - scen_c_profit

print("SCENARIO C: COMPOUNDED OPERATIONAL STRESS")
print(f"10% discount increase + 15% delivery risk increase across catalog.")
print(f"Projected Revenue:      ${scen_c_revenue:,.2f} (No change)")
print(f"Projected Profit:       ${scen_c_profit:,.2f} (Estimated)")
print(f"New Catalog Margin:     {scen_c_margin:.2%} (Drop of {((baseline_margin - scen_c_margin) * 100):.2f} pts)\n")

print("Strategic Takeaway:")
print(f"Allowing operational stagnation to compound directionally destroys an additional ${scen_c_profit_loss:,.2f} in net profit, eroding {margin_decay_rate:.2%} of the aggregate margin.")

SCENARIO C: COMPOUNDED OPERATIONAL STRESS
10% discount increase + 15% delivery risk increase across catalog.
Projected Revenue:      $35,214,428.98 (No change)
Projected Profit:       $3,625,796.46 (Estimated)
New Catalog Margin:     10.30% (Drop of 0.51 pts)

Strategic Takeaway:
Allowing operational stagnation to compound directionally destroys an additional $180,624.17 in net profit, eroding 4.75% of the aggregate margin.


**Business Interpretation:**
* **The Cost of Inaction:** Scenario C quantifies the slow bleed of chronic operational failure. While it doesn't instantly vaporize the bottom line like a shock to the core products (Scenario B), allowing the discount addiction and delivery failures to worsen by just 10-15% silently erodes over \\$180,000 in pure net profit.
* **A Controllable Variable:** Unlike a sudden supplier price hike, discount dependency and delivery SLAs are entirely within the business's control. This $180k loss is an unforced error. Continuing to operate with "business as usual" is a mathematical guarantee of margin decay.

## 7. Scenario Comparison Summary
Consolidate the baseline and all three stress-test scenarios into a single executive summary table. This provides the founder with a clear, quantified menu of risks and opportunities.

In [8]:
# Build the summary dataframe
summary_data = [
    {
        "Scenario": "Baseline (Current State)",
        "Action": "None",
        "Projected Revenue": baseline_revenue,
        "Projected Profit": baseline_profit,
        "Profit Impact ($)": 0,
        "Profit Retained (%)": 1.0000
    },
    {
        "Scenario": "Scenario A: Zombie SKU Elimination",
        "Action": "Cut 28 failing SKUs",
        "Projected Revenue": scen_a_revenue,
        "Projected Profit": scen_a_profit,
        "Profit Impact ($)": scen_a_profit - baseline_profit,
        "Profit Retained (%)": profit_retained_pct
    },
    {
        "Scenario": "Scenario B: Core SKU Disruption",
        "Action": "20% margin compression on top 8 SKUs",
        "Projected Revenue": scen_b_revenue,
        "Projected Profit": scen_b_profit,
        "Profit Impact ($)": -core_8_profit_loss,
        "Profit Retained (%)": 1 - profit_destroyed_pct
    },
    {
        "Scenario": "Scenario C: Compounded Stress",
        "Action": "10% more discounting + 15% more late deliveries",
        "Projected Revenue": scen_c_revenue,
        "Projected Profit": scen_c_profit,
        "Profit Impact ($)": -scen_c_profit_loss,
        "Profit Retained (%)": scen_c_profit / baseline_profit
    }
]

summary_df = pd.DataFrame(summary_data)

# Format for display
display_df = summary_df.copy()
display_df['Projected Revenue'] = display_df['Projected Revenue'].apply(lambda x: f"${x:,.0f}")
display_df['Projected Profit'] = display_df['Projected Profit'].apply(lambda x: f"${x:,.0f}")
display_df['Profit Impact ($)'] = display_df['Profit Impact ($)'].apply(lambda x: f"${x:,.0f}")
display_df['Profit Retained (%)'] = display_df['Profit Retained (%)'].apply(lambda x: f"{x:.2%}")

print("EXECUTIVE SCENARIO SUMMARY")
print(display_df.to_string(index=False))

# Export for Power BI
summary_df.to_csv('../data/processed/scenario_simulation_results.csv', index=False)
print("\nExported to: ../data/processed/scenario_simulation_results.csv")

EXECUTIVE SCENARIO SUMMARY
                          Scenario                                          Action Projected Revenue Projected Profit Profit Impact ($) Profit Retained (%)
          Baseline (Current State)                                            None       $35,214,429       $3,806,421                $0             100.00%
Scenario A: Zombie SKU Elimination                             Cut 28 failing SKUs       $34,450,987       $3,739,118          $-67,303              98.23%
   Scenario B: Core SKU Disruption            20% margin compression on top 8 SKUs       $35,214,429       $3,163,131         $-643,289              83.10%
     Scenario C: Compounded Stress 10% more discounting + 15% more late deliveries       $35,214,429       $3,625,796         $-180,624              95.25%

Exported to: ../data/processed/scenario_simulation_results.csv


In [10]:
core_8 = df.nlargest(8, 'total_profit')[['product_card_id', 'product_name', 'category_name', 
                                          'total_revenue', 'total_profit', 'net_profit_pct',
                                          'late_delivery_risk_rate', 'composite_risk_score']]
core_8.to_csv('../data/processed/core_profit_drivers.csv', index=False)
print(core_8.to_string(index=False))

 product_card_id                                  product_name        category_name  total_revenue  total_profit  net_profit_pct  late_delivery_risk_rate  composite_risk_score
            1004     Field & Stream Sportsman 16 Gun Fire Safe              Fishing     6637668.10     731576.18          0.1102                   0.5734                  51.3
             365              Perfect Fitness Perfect Rip Deck               Cleats     4233794.25     473820.89          0.1119                   0.5739                  51.3
             957 Diamondback Women's Serene Classic Comfort Bi     Camping & Hiking     3946836.86     409895.62          0.1039                   0.5691                  50.9
             191             Nike Men's Free 5.0+ Running Shoe     Cardio Equipment     3507549.21     359793.63          0.1026                   0.5692                  50.8
             502          Nike Men's Dri-FIT Victory Golf Polo      Women's Apparel     3011600.00     334335.53        

 ### Phase 4 Complete
The scenario simulations are complete and exported. The mathematical modeling proves that amputation is safe (Scenario A), concentration is a massive vulnerability (Scenario B), and operational stagnation is expensive (Scenario C). 

The analytical pipeline is now fully closed. All insights, validated metrics, and scenario data will now be synthesized into the final Executive Report and Power BI Dashboard.